In [4]:
import yaml
import copy
import numpy as np

from components.component import Component
from systems.series_system import SeriesSystem as SS
from agents.replacement_agent import ReplacementAgent as RA

### 1. Quick-start

First cell is a simple quick-start, where in the background the default configs specifying the components, the series system and the agent are loaded. With these configs, a series system with 3 identical components are instantiated (number of components is specified in system_config.yaml), which is then passed to the agent as an argument for instantiation. 

The final system is then subsequently evaluated with the rh1 policy, where the component-specific failure probability threshold is computed based on the preventive and corrective failure costs. Upon completion, the agent returns the system's long-running maintenance cost per unit time, its variance, as well as the component-specific maintenance cost rates and their corresponding variances.

In [2]:
S = SS()
A = RA(sys=S)
c_inf_sys, var_c_inf_sys, c_inf_comps, var_c_inf_comps = A.eval_rh(fun="rh1")

print(f"System cost rate & variance: C={c_inf_sys:.4f}, Var[C]={var_c_inf_sys:.3e}")
print(f"Component cost rates & variances: Ci={c_inf_comps}, Var[Ci]={var_c_inf_comps}")

Components all initialized with default config!
Step: 50
Step: 100
Step: 150
Step: 200
Step: 250
Step: 300
Step: 350
Step: 400
System cost rate & variance: C=0.2124, Var[C]=1.181e-06
Component cost rates & variances: Ci=[0.10579768 0.10663459], Var[Ci]=[5.77827011e-07 6.02865988e-07]


### 2. Component Customization

Now the next cell shows how you can create custom components. As an example, three components with different variable costs are created. 

1. The default configs are loaded, and the variable cost is specified per component
2. One loops over the variable cost vector, overwrites the config with the respective value and instantiates a Component object, which are collected in a list
3. This vector of components is then passed as an argument to the series system, which is in turn again passed as an argument to the agent
4. Finally, the system is again evaluated with the rh1 policy

In [3]:
# load the default
with open("./systems/system_config.yaml", "r") as file:
    sys_cf = yaml.load(file, Loader=yaml.FullLoader)
with open("./components/component_config.yaml", "r") as file:
    comp_cf = yaml.load(file, Loader=yaml.FullLoader)

cv_list = [5, 10, 15]
comp_list = []
for i, cv in enumerate(cv_list):
    comp_cf["c_v"] = cv
    print(f"Component {i} config: ", comp_cf)
    comp_list.append(Component(cf=copy.deepcopy(comp_cf)))
    
# create sys & Agent with comps
S = SS(comps=comp_list)
A = RA(sys=S)

c_inf_sys, var_c_inf_sys, c_inf_comps, var_c_inf_comps = A.eval_rh(fun="rh1")
print(f"System cost rate & variance: C={c_inf_sys:.4f}, Var[C]={var_c_inf_sys:.3e}")
print(f"Component cost rates & variances: Ci={c_inf_comps}, Var[Ci]={var_c_inf_comps}")

Component 0 config:  {'rng_seed': 2026, 'Delta_T': 10, 'c_v': 5, 'c_c': 100, 'B': 500, 'dist_type': 'normal', 'mu_TF': 225, 'sigma_TF': 40, 'sig_int_factor': 15, 'corr_len': 50, 'sigma_lnerr': 0.4}
Component 1 config:  {'rng_seed': 2026, 'Delta_T': 10, 'c_v': 10, 'c_c': 100, 'B': 500, 'dist_type': 'normal', 'mu_TF': 225, 'sigma_TF': 40, 'sig_int_factor': 15, 'corr_len': 50, 'sigma_lnerr': 0.4}
Component 2 config:  {'rng_seed': 2026, 'Delta_T': 10, 'c_v': 15, 'c_c': 100, 'B': 500, 'dist_type': 'normal', 'mu_TF': 225, 'sigma_TF': 40, 'sig_int_factor': 15, 'corr_len': 50, 'sigma_lnerr': 0.4}
Step: 50
Step: 100
Step: 150
Step: 200
Step: 250
Step: 300
Step: 350
Step: 400
System cost rate & variance: C=0.3159, Var[C]=1.780e-06
Component cost rates & variances: Ci=[0.07991226 0.1053021  0.1307057 ], Var[Ci]=[5.05980667e-07 5.95635215e-07 6.78640455e-07]


### 3. Limited sample sizes

The last cell shows how you can do a policy evaluation, if you have a set of failure samples available for each component.
Here, the batch size of the system is set to 1, the tf samples of each component are simply appended, until the first component has replenished all samples. 
Exemplarily, the simulation of a 3-component series system is shown with 10 failure samples each.

In [ ]:
with open("./systems/system_config.yaml", "r") as file:
    sys_cf = yaml.load(file, Loader=yaml.FullLoader)
with open("./components/component_config.yaml", "r") as file:
    comp_cf = yaml.load(file, Loader=yaml.FullLoader)

C = Component()

comp_list = []
for k in range(3):
    tf_samples = C.draw_tf_samples_bigger_val(Ns=10, val=sys_cf["Delta_T"])
    comp_list.append(
        Component(
            cf=comp_cf,
            reset_flag=True,
            tf_samples=tf_samples,
        )
    )

# create sys & Agent with comps
S = SS(comps=comp_list)
A = RA(sys=S)

c_inf_sys, var_c_inf_sys, c_inf_comps, var_c_inf_comps = A.eval_rh(fun="rh1")
print(f"System cost rate & variance: C={c_inf_sys:.4f}, Var[C]={var_c_inf_sys:.3e}")
print(f"Component cost rates & variances: Ci={c_inf_comps}, Var[Ci]={var_c_inf_comps}")

Step: 50
Step: 100
Step: 150
Step: 200
System cost rate & variance: C=0.2940, Var[C]=1.561e-03
Component cost rates & variances: Ci=[0.12442396 0.08374384 0.08585859], Var[Ci]=[1.51583565e-03 2.74978349e-05 1.78170634e-05]
